In [115]:
import os

import pandas

from helpers.data import MinioHelper, CalcHelper
import csv
from dotenv import load_dotenv

load_dotenv(override=True)

SCORING = {
    'td': 6,
    'rushingyards': 0.1,
    'receivingyards': 0.1,
    'passingyards': 0.05,
    'receptions': 1
}

YEARS = ['2025']
SEASON = 'regular'
TYPE = 'players'
BUCKET = 'football-warehouse'
PLAYER_NAME = 'Patrick Mahomes'
FOCUS_YEAR = 2023

RANKING_FILE = os.getenv('RANKING_FILE')
OUTPUT_FILE = os.getenv('OUTPUT_FILE')

ranking_frame = pandas.read_csv(RANKING_FILE, sep=',', quoting=csv.QUOTE_ALL)



In [116]:
helper = MinioHelper(BUCKET)

data = {}

for year in YEARS:
    data[year] = helper.get_statistics([year], SEASON, TYPE)
    

In [117]:
frames = []

for i in range(0, len(YEARS)):
    year = YEARS[i]
    if year == YEARS[-1]:
        continue
    prev_year = YEARS[i+1]
    left_frame = CalcHelper.calculate_points(data[year], SCORING)
    right_frame = CalcHelper.calculate_points(data[prev_year], SCORING)
    result = CalcHelper.calculate_differential(left_frame, right_frame, year)
    frames.append(result)
    
stats_frame = pandas.concat(frames, ignore_index=True)
stats_frame = stats_frame.fillna(0, axis=1)
    

In [118]:
clean_frame = stats_frame.loc[(stats_frame['player_name'] != 'Team') & (stats_frame['points'] != 0)]


In [119]:
grouped_frame = clean_frame.groupby(['player_url', 'player_name', 'year'])['scoring_diff'].sum().reset_index()

In [120]:
yr_frame = stats_frame.loc[(stats_frame['year'] == FOCUS_YEAR) & (stats_frame['points'] > 0) & (stats_frame['player_name'] != 'Team')]

In [121]:
pts_frame = yr_frame.groupby(['player_url', 'player_name'])['points'].mean().reset_index()

In [122]:
diff_year_frame = grouped_frame.loc[grouped_frame['year'] == FOCUS_YEAR]

In [123]:
joined_frame = diff_year_frame.merge(pts_frame, how='inner', suffixes=(None, '_r'), on=['player_url'])

In [124]:
joined_frame = joined_frame.drop(labels=['player_name_r'], axis=1)

In [125]:
joined_with_rank = ranking_frame.merge(joined_frame, how='left', right_on='player_name', left_on='PLAYER NAME', suffixes=(None, '_r'))

In [126]:
joined_with_rank.to_csv(OUTPUT_FILE, sep=',', quoting=csv.QUOTE_ALL)